# v9 node dial — geometry, performance, and an honest choice of alpha

The dial moves the graph's **input**:

$$x_{\text{prot}}(\alpha)\;=\;\mu\;+\;\alpha\cdot\text{centred}(\text{ESM})\;+\;(1-\alpha)\cdot\text{centred}(\text{identity vector})$$

one fixed near-orthogonal random vector per receptor, drawn once by name. So

| alpha | what the receptor node is | the model it equals |
|---|---|---|
| **0** | identity alone — who this receptor is, nothing about what it looks like | the one-hot graph |
| **1** | the embedding file itself, unchanged | the legacy graph |

**The direction is the OPPOSITE of the v8 gate**, where alpha ran structure → function.
Never put the two in one frame: `nodes` separates them and the loader defaults keep them
apart.

### The three questions, in order

1. **Geometry** — does the receptor cloud actually travel between the two ideological
   ends? ESM on one side, the train response profile on the other, three second-order
   measures, each against its own permutation null.
2. **Performance** — what the metrics and the mean places do along the dial.
3. **The choice** — which alpha to report. Made on **validation**, then and only then
   carried to test.

### Why step 3 is built the way it is

Alpha is a hyperparameter: fixed before training, used unchanged at inference. Choosing
it by the score on the folds the paper reports is selection on the test set — the
winner's margin is then partly the luck of those folds, and quoting it as if alpha had
been fixed in advance overstates the claim.

The sweep writes **test only**. `scripts/analysis/val_rescore.py` fills the gap without
retraining anything: it reloads each cell's dumped receptor cloud, refits the boosting
head on the same train rows with the same seed — reproducing the estimator whose test
score is already on disk — and asks it for the validation rows instead. Run it once
before this notebook:

```sh
python scripts/analysis/val_rescore.py --root results/graph/v9_node_dial --dry-run
python scripts/analysis/val_rescore.py --root results/graph/v9_node_dial
```

Everything below is drawn from `scripts/analysis/alpha_grid.py` and
`scripts/analysis/alpha_choice.py`. **This notebook computes nothing of its own** — if a
number here disagrees with the CLI, the notebook is the one that is wrong.

In [ ]:
# ------------------------------------------------------------------ KNOBS ---
ROOT_DIR   = "results/graph/v9_node_dial"   # the sweep directory
MOL_SOURCE = "chemberta"                    # chemberta | gin | ecfp | None = all
DATASET    = None                           # None = every dataset on disk
REGIME     = None                           # None = both regimes
SEED       = None                           # None = every model seed present
CI_LEVEL   = 0.95                           # Student-t, over the splits
GEOM_SCALE = "value"                        # "value" = the measure, "z" = vs its null
METRICS    = None                           # None = the whole battery for the task
SHOW_PLACES_FOR = None                      # None = every metric available
SAVE_FIGS  = False
FIG_DIR    = "results/graph/v9_node_dial/figures"

NODES = "nodedial"        # do NOT widen this: alpha means the opposite on the v8 gate

In [ ]:
import pathlib, sys, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

_root = pathlib.Path.cwd()
while not (_root / "pyproject.toml").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))

from scripts.analysis import alpha_grid as ag
from scripts.analysis import alpha_choice as ac

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200, "display.max_columns", 60)

KW = dict(root=ROOT_DIR, mol_source=MOL_SOURCE, nodes=NODES,
          dataset=DATASET, regime=REGIME, seed=SEED)

test = ag.load(split="test", **KW)
try:
    val = ag.load(split="val", **KW)
except SystemExit:
    val = None

MET = ag.metric_for(test)                       # metric of record per dataset
SERIES = sorted(test.series.unique())
print(f"{len(SERIES)} series, metric of record {MET}")
if val is None:
    print("\n!! NO VALIDATION FRAME -- block 3 will not run.\n"
          f"   python scripts/analysis/val_rescore.py --root {ROOT_DIR} --dry-run\n"
          f"   python scripts/analysis/val_rescore.py --root {ROOT_DIR}")
ag.coverage(test)

## Gate: is the dial anchored, and how small a difference can it resolve?

Two checks before any curve is read.

**The anchor.** At alpha=1 the mixture returns the embedding file unchanged, so that end
*is* the legacy graph and must land on the `graph_legacy` arm. A gap wider than the
tolerance means the end is not what it claims and every position inherits the fault.

**The repeat floor.** Those same two rows are one model trained twice — `_train_one`
never applies `hp["seed"]`, so the graph's initialisation comes from the global RNG. Their
gap is therefore a free measurement of this cell's run-to-run noise, and it is the ruler
for every difference along the dial. It is **not** the across-fold interval: that one is
shared by every row and largely cancels between two of them.

In [ ]:
anchor = ag.anchor_check(test)
display(anchor.round(4))
if len(anchor) and not anchor["ok"].all():
    print("!! an end does not reproduce its reference -- read no curve below until "
          "that is explained")

FLOOR = {r.series: abs(r.gap) for r in anchor.itertuples()}
if FLOOR:
    print(f"\nrepeat floor per series (|alpha=1 - legacy| on the metric of record):")
    for s, v in FLOOR.items():
        print(f"  {s:<34} {v:.4f}")
    print(f"  {'POOLED':<34} {np.mean(list(FLOOR.values())):.4f}"
          "   <-- the size a difference must clear to be readable")

In [ ]:
# ------------------------------------------------------- plotting helpers ---
C = {"esm": "#1f6fb4", "fun": "#c9531f", "dial": "#1f6fb4", "val": "#c9531f",
     "boost_full": "#6b7280", "graph_legacy": "#7a4fa3", "naive": "#b9bec6",
     "ink": "#1f2328", "muted": "#6b7280", "grid": "#e8e8e4"}
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white",
                     "font.size": 9, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.edgecolor": "#d7d7d2"})

def panels(nrow, ncol, w=3.5, h=2.7, **kw):
    fig, ax = plt.subplots(nrow, ncol, figsize=(w * ncol, h * nrow), squeeze=False, **kw)
    return fig, ax

def band(ax, d, color, label=None, lw=2):
    """A curve with its across-fold interval. The interval is context, not the test:
    every row shares the folds, so overlapping bands do NOT mean 'no difference'."""
    if d.empty:
        return
    d = d.sort_values("alpha")
    x = d["alpha"].to_numpy(float)
    ax.fill_between(x, d["lo"], d["hi"], color=color, alpha=0.10, lw=0)
    ax.plot(x, d["mean"], color=color, lw=lw, label=label, zorder=3)
    ax.plot(x, d["mean"], "o", color=color, ms=4, mec="white", mew=1, zorder=4)

def floor_bar(ax, series, x=-0.09):
    """The repeat gap as an I-beam: the smallest difference this cell can resolve."""
    g = FLOOR.get(series)
    if not g:
        return
    y0, y1 = ax.get_ylim()
    mid = y0 + 0.42 * (y1 - y0)
    ax.plot([x, x], [mid - g / 2, mid + g / 2], color=C["ink"], lw=1.6, clip_on=False)
    for yy in (mid - g / 2, mid + g / 2):
        ax.plot([x - 0.015, x + 0.015], [yy, yy], color=C["ink"], lw=1.6, clip_on=False)
    ax.annotate(f"repeat\n{g:.3f}", xy=(x, mid + g / 2), xytext=(0, 5),
                textcoords="offset points", fontsize=7, ha="center", va="bottom",
                color=C["ink"], annotation_clip=False, linespacing=1.15)
    ax.set_xlim(x - 0.05, 1.0)

def alpha_axis(ax, bottom=True):
    ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.grid(axis="y", color=C["grid"], lw=0.8)
    ax.set_axisbelow(True)
    if bottom:
        ax.set_xlabel("alpha    0 = identity alone  ->  1 = ESM nodes")

def figlegend(fig, ax, ncol=4):
    h, l = ax.get_legend_handles_labels()
    if not h:
        return
    fig.tight_layout()
    fig.subplots_adjust(bottom=fig.subplotpars.bottom + 0.62 / fig.get_figheight())
    fig.legend(h, l, loc="lower center", ncol=ncol, frameon=False, fontsize=9)

def note_empty(ax, msg="not run yet"):
    ax.text(0.5, 0.5, msg, transform=ax.transAxes, ha="center", va="center",
            color=C["muted"], fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

def savefig(fig, name):
    if not SAVE_FIGS:
        return
    d = pathlib.Path(FIG_DIR); d.mkdir(parents=True, exist_ok=True)
    fig.savefig(d / f"{name}.png", dpi=160, bbox_inches="tight")
    print(f"  -> {d / (name + '.png')}")

---
# 1 · Geometry: does the cloud travel between the two ends?

Two reference clouds, both fit on **train rows only**:

- **ESM** — the raw protein embedding. Structure.
- **response profile** — the train receptor × train-odorant response matrix, row-centred.
  Function. No test odorant enters it; a diagnostic that peeked would report the leak as
  a success.

Three second-order measures, all reported so that **larger is more aligned** (Procrustes
as 1 − disparity): RSA compares neighbour orderings, CCA the shared linear subspace,
Procrustes the shape after the best rigid fit.

The dial working means `vs ESM` **rises** with alpha and `vs response` falls — the v9
direction, opposite to v8.

> On **M2OR read the `fun` column as an upper bound**. That matrix is sparse, so most
> cells are "not assayed" rather than "no response", and the imputation writes the global
> mean into them. What survives is partly *which pairs were measured* — assay design, not
> binding. The `tested mask` control put that share at 76–103% of the graph's own gain on
> M2OR. On the complete insect panels it does not arise.

In [ ]:
geo = ag.geometry(test, scale=GEOM_SCALE)
fig, AX = panels(len(SERIES), len(ag.GEOMS), h=2.6)
for i, s in enumerate(SERIES):
    for j, g in enumerate(ag.GEOMS):
        ax = AX[i][j]
        got = False
        for ref in ag.REFS:
            d = geo[(geo.series == s) & (geo.geom == g) & (geo.ref == ref)]
            if not d.empty:
                got = True
                band(ax, d, C[ref], label=ag.REF_LABEL[ref] if i == j == 0 else None)
        if not got:
            note_empty(ax)
            continue
        alpha_axis(ax, bottom=(i == len(SERIES) - 1))
        if i == 0:
            ax.set_title(ag.GEOM_LABEL[g], fontsize=10)
        if j == 0:
            ax.set_ylabel(s, fontsize=9)
fig.suptitle(f"1.1  alignment of the receptor cloud with each end  [{GEOM_SCALE}]",
             fontsize=12, y=1.002)
figlegend(fig, AX[0][0], ncol=2)
savefig(fig, "1_1_geometry_curves")

In [ ]:
# 1.2  where the two alignments cross, per (series, measure). Each reference curve is
# min-max scaled inside its own panel first -- raw RSA against ESM and against a
# response matrix are not on one scale, so their literal intersection would be an
# artefact of that.
cx = ag.crossover(geo)
if cx.empty or cx["alpha_cross"].isna().all():
    print("no crossing on any panel: the two alignments never swap order over the dial "
          "-- which is itself the finding, not a failure")
else:
    fig, ax = panels(1, 1, w=7, h=0.5 + 0.42 * len(SERIES))
    ax = ax[0][0]
    ys = {s: k for k, s in enumerate(SERIES)}
    for g, mk in zip(ag.GEOMS, ["o", "s", "D"]):
        d = cx[cx.geom == g].dropna(subset=["alpha_cross"])
        ax.plot(d["alpha_cross"], [ys[s] for s in d.series], mk, ms=8, mec="white",
                mew=1.2, label=ag.GEOM_LABEL[g], color=C["dial"], alpha=0.75)
    ax.set_yticks(range(len(SERIES))); ax.set_yticklabels(SERIES, fontsize=9)
    ax.set_xlim(-0.02, 1.02); ax.set_xlabel("alpha at which structure overtakes function")
    ax.grid(axis="x", color=C["grid"], lw=0.8); ax.set_axisbelow(True)
    ax.set_title("1.2  the crossover", fontsize=12, loc="left")
    ax.legend(frameon=False, fontsize=8, ncol=3, loc="upper center",
              bbox_to_anchor=(0.5, -0.35))
    savefig(fig, "1_2_crossover")
display(cx.round(3))

In [ ]:
# 1.3  how far the cloud actually travels. Per (series, measure, reference): the value
# at each END of the dial, as a dumbbell. A short bar means the geometry does not move
# -- and then no performance difference along the dial can be attributed to it.
rows = []
for (s, g, r), d in geo.groupby(["series", "geom", "ref"]):
    d = d.sort_values("alpha")
    if len(d) < 2:
        continue
    rows.append(dict(series=s, geom=g, ref=r, a0=float(d["mean"].iloc[0]),
                     a1=float(d["mean"].iloc[-1]),
                     travel=float(d["mean"].iloc[-1] - d["mean"].iloc[0])))
trav = pd.DataFrame(rows)
if trav.empty:
    print("no geometry to travel")
else:
    fig, AX = panels(1, len(ag.GEOMS), w=3.6, h=0.6 + 0.42 * len(SERIES))
    for j, g in enumerate(ag.GEOMS):
        ax = AX[0][j]
        d = trav[trav.geom == g]
        lab, y = [], 0
        for s in SERIES:
            for r in ag.REFS:
                q = d[(d.series == s) & (d.ref == r)]
                if q.empty:
                    continue
                a0, a1 = float(q.a0.iloc[0]), float(q.a1.iloc[0])
                ax.plot([a0, a1], [y, y], color=C[r], lw=2, alpha=0.55, zorder=2)
                ax.plot([a0], [y], "o", color=C[r], ms=6, mec="white", mew=1, zorder=3)
                ax.plot([a1], [y], "D", color=C[r], ms=6, mec="white", mew=1, zorder=3)
                lab.append(f"{s.split(' / ')[0]}/{s.split(' / ')[-1][:5]} {r}")
                y += 1
        ax.set_yticks(range(len(lab))); ax.set_yticklabels(lab, fontsize=7)
        ax.set_title(ag.GEOM_LABEL[g], fontsize=9)
        ax.grid(axis="x", color=C["grid"], lw=0.8); ax.set_axisbelow(True)
        ax.invert_yaxis()
    fig.suptitle("1.3  travel from alpha=0 (circle) to alpha=1 (diamond)",
                 fontsize=12, y=1.02)
    fig.tight_layout()
    savefig(fig, "1_3_travel")
    display(trav.pivot_table(index=["series", "ref"], columns="geom",
                             values="travel").round(3))

---
# 2 · Performance along the dial

The panels below are read against the **repeat bar**, not against the shaded interval.
The shaded interval is the across-fold spread: it is 2–30× wider than the whole dial on
these cells, and every competitor shares the folds that produce it, so two overlapping
bands say nothing about whether two positions differ.

In [ ]:
# 2.1  the metric of record, with boost as a reference line
fig, AX = panels((len(SERIES) + 1) // 2, 2, h=2.9)
axes = [a for row in AX for a in row]
for ax, s in zip(axes, SERIES):
    ds = test[test.series == s].dataset.iloc[0]
    m = MET[ds]
    band(ax, ag.curve(test[test.series == s], m, level=CI_LEVEL), C["dial"])
    lv = ag.levels(test[test.series == s], m, level=CI_LEVEL)
    for arm in ["boost_full", "graph_legacy"]:
        q = lv[lv.arm == arm]
        if q.empty:
            continue
        v = float(q["mean"].iloc[0])
        ax.axhline(v, color=C[arm], lw=1.4, ls=(0, (5, 3)), zorder=2)
        ax.annotate(ag.ARM_LABEL[arm], xy=(0.30, v), xycoords=("axes fraction", "data"),
                    xytext=(0, 4), textcoords="offset points", fontsize=8, color=C[arm])
    ax.set_title(s, fontsize=10, loc="left"); ax.set_ylabel(m)
    alpha_axis(ax); floor_bar(ax, s)
for ax in axes[len(SERIES):]:
    ax.axis("off")
fig.suptitle("2.1  metric of record along the dial", fontsize=12, y=1.002)
fig.tight_layout()
savefig(fig, "2_1_metric_of_record")

In [ ]:
# 2.2  the PAIRED difference against boost. Both arms ran on the same split with the
# same draw, so differencing removes the split and the draw at once -- this is the
# panel that can actually resolve a 0.01 effect, and 2.1 is not.
fig, AX = panels((len(SERIES) + 1) // 2, 2, h=2.7)
axes = [a for row in AX for a in row]
for ax, s in zip(axes, SERIES):
    ds = test[test.series == s].dataset.iloc[0]
    d = ag.delta_vs(test[test.series == s], MET[ds], level=CI_LEVEL)
    if d.empty:
        note_empty(ax); ax.set_title(s, fontsize=10, loc="left"); continue
    band(ax, d, C["dial"])
    ax.axhline(0, color=C["boost_full"], lw=1.4, ls=(0, (5, 3)))
    ax.set_title(s, fontsize=10, loc="left")
    ax.set_ylabel(f"{MET[ds]}  (dial - boost, paired)")
    alpha_axis(ax)
for ax in axes[len(SERIES):]:
    ax.axis("off")
fig.suptitle("2.2  paired advantage over boost -- above zero is a win",
             fontsize=12, y=1.002)
fig.tight_layout()
savefig(fig, "2_2_paired_delta")

In [ ]:
# 2.3  the whole battery: does the story survive the metric?
for s in SERIES:
    sub = test[test.series == s]
    ds = sub.dataset.iloc[0]
    mets = METRICS or ag.metrics_available(test, ds)
    fig, AX = panels(1, len(mets), w=3.1, h=2.5)
    for j, m in enumerate(mets):
        ax = AX[0][j]
        band(ax, ag.curve(sub, m, level=CI_LEVEL), C["dial"])
        lv = ag.levels(sub, m, level=CI_LEVEL)
        q = lv[lv.arm == "boost_full"]
        if not q.empty:
            ax.axhline(float(q["mean"].iloc[0]), color=C["boost_full"], lw=1.3,
                       ls=(0, (5, 3)))
        ax.set_title(m + ("  (lower is better)" if m in ag.LOWER_IS_BETTER else ""),
                     fontsize=9)
        alpha_axis(ax, bottom=(j == 0))
    fig.suptitle(f"2.3  {s} -- every metric, grey dashed = boost", fontsize=11, y=1.04)
    fig.tight_layout()
    savefig(fig, f"2_3_battery_{s.replace(' / ', '_').replace(' ', '')}")

In [ ]:
# 2.4  MEAN PLACE per metric. Ranked within each split and averaged, 1 = best -- the
# summary that cannot be moved by one hard fold, unlike a curve of means. The reference
# arms are in the race, so a place also says whether the dial beats boost at all.
for s in SERIES:
    sub = test[test.series == s]
    ds = sub.dataset.iloc[0]
    mets = SHOW_PLACES_FOR or METRICS or ag.metrics_available(test, ds)
    fig, AX = panels(1, len(mets), w=3.1, h=2.5)
    for j, m in enumerate(mets):
        ax = AX[0][j]
        p = ag.places(sub, m)
        if p.empty:
            note_empty(ax); continue
        dial = p[p.alpha.notna()].sort_values("alpha")
        ax.plot(dial["alpha"], dial["place"], color=C["dial"], lw=2, zorder=3)
        ax.plot(dial["alpha"], dial["place"], "o", color=C["dial"], ms=4,
                mec="white", mew=1, zorder=4)
        for arm, lab in (("boost [ESM || mol]", "boost_full"),
                         ("legacy graph", "graph_legacy")):
            q = p[p.competitor == arm]
            if q.empty:
                continue
            ax.axhline(float(q["place"].iloc[0]), color=C[lab], lw=1.3, ls=(0, (5, 3)))
        ax.invert_yaxis()            # 1 = best, so up is better
        ax.set_title(f"{m}   (of {int(p['k'].iloc[0])})", fontsize=9)
        ax.set_ylabel("mean place" if j == 0 else "")
        alpha_axis(ax, bottom=(j == 0))
    fig.suptitle(f"2.4  {s} -- mean place, 1 = best (axis inverted, up is better)",
                 fontsize=11, y=1.04)
    fig.tight_layout()
    savefig(fig, f"2_4_places_{s.replace(' / ', '_').replace(' ', '')}")

In [ ]:
# 2.5  the numbers behind 2.1-2.4
for s in SERIES:
    sub = test[test.series == s]
    ds = sub.dataset.iloc[0]
    m = MET[ds]
    cur = ag.curve(sub, m, level=CI_LEVEL)[["alpha", "mean", "hw", "n"]]
    dlt = ag.delta_vs(sub, m, level=CI_LEVEL)[["alpha", "mean", "hw", "won", "n"]]
    plc = ag.places(sub, m)
    plc = plc[plc.alpha.notna()][["alpha", "place"]]
    out = (cur.rename(columns={"mean": m, "hw": "+/-", "n": "splits"})
           .merge(dlt.rename(columns={"mean": "d vs boost", "hw": "d +/-",
                                      "n": "paired"}), on="alpha", how="left")
           .merge(plc, on="alpha", how="left"))
    print(f"\n=== {s}   [{m}]   repeat floor {FLOOR.get(s, float('nan')):.4f}")
    display(out.round(4))

---
# 3 · Choosing alpha — on validation, and only then on test

The rules this block enforces:

1. **The search runs on validation rows.** They were never used to train the graph
   (message-passing edges come from train only, the epoch count is fixed, val is never
   consulted) and never used to fit the head. They are also the same *shape* as test —
   `inductive_molecule_v5` draws val molecules disjoint from train and test, and
   `our_inductive` spreads val over the same dynamic-range order — so a cold-molecule
   choice is made on cold-molecule evidence.
2. **The 1-SE rule.** Alphas within one standard error of the leader are
   indistinguishable from it. An argmax out of a flat set is noise-chasing.
3. **Test is read once**, at the chosen alpha, and the gap to the best-on-test alpha is
   printed as the *optimism avoided* — exactly what choosing on test would have added to
   the claim.

In [ ]:
import argparse
if val is None:
    print("no validation frame -- see the load cell for the command that builds it")
else:
    ARGS = argparse.Namespace(metric=None, loo=False, select_on="val", at_alpha=None)
    metric_of = ac._metric_of(val, ARGS)
    sc_val = ac.scores(val, metric_of)
    rk_val = ac.ranked(sc_val)
    tied, cut = ac.one_se(rk_val)
    best, _ = ac.chosen_alpha(val, metric_of)
    display(rk_val.round(3))
    print(f"\nBEST on validation: alpha = {best:g}   (mean rank "
          f"{float(rk_val.iloc[0].mean_rank):.2f})")
    print(f"WITHIN 1 SE of it (<= {cut:.2f}): "
          + ", ".join(f"{a:g}" for a in sorted(tied.alpha.dropna())))
    if len(tied) > 1:
        print("   -> the dial is FLAT there. Fix alpha at the end you can argue for and "
              "say so;\n      this table is the evidence that nothing was lost by it.")

In [ ]:
# 3.2  the validation leaderboard as a curve. The 1-SE band is the honest width of the
# choice: everything inside it is one answer, not a ranking.
if val is not None:
    gate = rk_val[rk_val.alpha.notna()].sort_values("alpha")
    fig, ax = panels(1, 1, w=7.2, h=3.2)
    ax = ax[0][0]
    ax.axhspan(float(rk_val[rk_val.alpha.notna()].mean_rank.min()), cut,
               color=C["dial"], alpha=0.10, lw=0, label="within 1 SE of the best")
    ax.errorbar(gate["alpha"], gate["mean_rank"], yerr=gate["se"], fmt="none",
                ecolor=C["dial"], alpha=0.5, elinewidth=1.2, capsize=3)
    ax.plot(gate["alpha"], gate["mean_rank"], color=C["dial"], lw=2, zorder=3)
    ax.plot(gate["alpha"], gate["mean_rank"], "o", color=C["dial"], ms=5,
            mec="white", mew=1, zorder=4)
    for arm, key in (("boost", "boost_full"), ("legacy", "graph_legacy")):
        q = rk_val[rk_val.competitor == arm]
        if not q.empty:
            v = float(q.mean_rank.iloc[0])
            ax.axhline(v, color=C[key], lw=1.4, ls=(0, (5, 3)))
            ax.annotate(arm, xy=(0.02, v), xycoords=("axes fraction", "data"),
                        xytext=(0, 4), textcoords="offset points",
                        fontsize=8, color=C[key])
    ax.axvline(best, color=C["val"], lw=1.4)
    ax.annotate(f"chosen  alpha={best:g}", xy=(best, ax.get_ylim()[1]),
                xytext=(4, -12), textcoords="offset points", color=C["val"], fontsize=9)
    ax.invert_yaxis()
    ax.set_ylabel("mean rank across cells (1 = best)")
    alpha_axis(ax)
    ax.set_title("3.2  the choice, on VALIDATION", fontsize=12, loc="left")
    ax.legend(frameon=False, fontsize=8)
    savefig(fig, "3_2_val_leaderboard")

In [ ]:
# 3.3  val against test, curve by curve. Not a check of the choice -- it is a check of
# whether validation is a usable GUIDE here at all: if the two disagree about the shape,
# the dial is inside the noise on this cell and no split can choose within it.
if val is not None:
    fig, AX = panels((len(SERIES) + 1) // 2, 2, h=2.9)
    axes = [a for row in AX for a in row]
    for ax, s in zip(axes, SERIES):
        ds = test[test.series == s].dataset.iloc[0]
        m = MET[ds]
        band(ax, ag.curve(test[test.series == s], m, level=CI_LEVEL), C["dial"],
             label="test" if ax is axes[0] else None)
        vs = val[val.series == s]
        if not vs.empty:
            band(ax, ag.curve(vs, m, level=CI_LEVEL), C["val"],
                 label="validation" if ax is axes[0] else None)
        ax.axvline(best, color=C["muted"], lw=1, ls=":")
        ax.set_title(s, fontsize=10, loc="left"); ax.set_ylabel(m)
        alpha_axis(ax)
    for ax in axes[len(SERIES):]:
        ax.axis("off")
    fig.suptitle("3.3  validation vs test -- dotted line is the chosen alpha",
                 fontsize=12, y=1.002)
    figlegend(fig, axes[0], ncol=2)
    savefig(fig, "3_3_val_vs_test")

In [ ]:
# 3.4  THE ONLY PLACE TEST IS READ. Per cell: the value at the chosen alpha, its paired
# advantage over boost, and how far the luckiest alpha on these very folds would have
# outscored it -- the optimism that choosing on test would have quietly added.
if val is not None:
    ac.confirm(val, test, argparse.Namespace(metric=None, at_alpha=None))

---
## What can be written down from this

Fill these in from the blocks above; each is a sentence the notebook has evidence for.

- **Does the cloud move?** 1.3 gives the travel per measure and reference. If it is
  short, the dial changes the input without changing the receptor geometry, and no
  performance difference along it can be attributed to structure.
- **Does the score move?** 2.1 against the repeat bar, and 2.2 for the paired version.
  A difference smaller than the bar is not resolvable at one model seed.
- **Which alpha, and does it matter?** 3.1 gives the leader and the 1-SE set. If the set
  is wide, the honest sentence is *"alpha was fixed at the end of the dial on the
  argument, not tuned; the sweep is the evidence that nothing was lost"*.
- **What does it cost to be honest?** The `best there` column in 3.4. That number is the
  margin a test-chosen alpha would have claimed and this protocol gives up.

Anything else — three datasets against each other, the molecule sources side by side —
belongs to `paper_tables.py`, which is the scoreboard of record. This notebook is the
shape of one dial.